# IMC metadata and training-overlap audit
CPU only, internet off. Attach ongoing competition inputs. No API credentials or downloads. This notebook reads training-image thumbnails only for provenance screening, not model features. Test-image bytes are never read. No predictions or splits are generated.


In [ ]:
"""Offline CPU metadata audit. Embedded verbatim in the portable notebook."""
import csv
import hashlib
import json
import os
import platform
import sqlite3
import tempfile
import zipfile
from collections import Counter
from contextlib import closing
from datetime import datetime, timezone
from pathlib import Path
import numpy as np

FIELDS = ['dataset', 'scene', 'image', 'rotation_matrix', 'translation_vector']
MAX_FILES = 1000000
MAX_ROWS = 1000000
MAX_METADATA_BYTES = 10 * 1024**2
MAX_INVENTORY_RECORDS = 20000

def dump(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')

def header(path):
    try:
        with path.open(encoding='utf-8-sig', newline='') as f:
            return next(csv.reader(f), [])
    except (OSError, UnicodeError, csv.Error):
        return []

def discover(input_root, override=None):
    """Bounded shallow metadata discovery, independent of mount names."""
    input_root = Path(input_root).resolve()
    if override:
        chosen = Path(override).resolve()
        if not chosen.is_relative_to(input_root) or not chosen.is_dir():
            raise ValueError('INPUT_PATH must be an attached directory beneath input_root')
        return chosen, [str(chosen)], 'manual'
    candidates = set()
    visited = 0
    for directory, dirs, files in os.walk(input_root, followlinks=False):
        visited += 1
        if visited > 10000:
            raise ValueError('Discovery limit reached. Set INPUT_PATH explicitly.')
        base = Path(directory)
        dirs[:] = [d for d in dirs if not (base/d).is_symlink() and len((base/d).relative_to(input_root).parts) <= 5]
        for name in files:
            p = base/name
            if p.suffix.lower() == '.csv' and not p.is_symlink() and p.stat().st_size <= MAX_METADATA_BYTES:
                h = header(p)
                if set(FIELDS).issubset(h) and ((base/'train').is_dir() or (base/'test').is_dir()):
                    candidates.add(base)
    candidates = sorted(candidates)
    if len(candidates) != 1:
        raise ValueError(f'Found {len(candidates)} candidate roots. Attach competition data or set INPUT_PATH explicitly. Candidates: {[str(p) for p in candidates]}')
    return candidates[0], [str(p) for p in candidates], 'automatic structure heuristic; provenance requires user confirmation'

def pose_state(row):
    try:
        r = np.array([float(x) for x in row['rotation_matrix'].split(';')]).reshape(3, 3)
        t = np.array([float(x) for x in row['translation_vector'].split(';')]).reshape(3)
        if np.isnan(r).all() and np.isnan(t).all(): return 'all_nan'
        if not np.isfinite(r).all() or not np.isfinite(t).all(): return 'nonfinite'
        if not np.allclose(r.T@r, np.eye(3), atol=1e-3) or abs(np.linalg.det(r)-1) > 1e-3: return 'invalid_rotation'
        return 'valid'
    except (ValueError, KeyError, AttributeError):
        return 'malformed'

def analyze_csv(path, db, role):
    counts, poses, issues = Counter(), Counter(), Counter()
    db.execute('drop table if exists records')
    db.execute('create table records(dataset text,image text,scene text,primary key(dataset,image))')
    rows = 0
    complete = True
    with path.open(encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        h = reader.fieldnames
        if h != FIELDS:
            return {'status':'unsupported_header', 'columns':h, 'complete':False}, counts
        for row in reader:
            if rows >= MAX_ROWS:
                complete = False; break
            rows += 1
            if None in row or any(row.get(k) is None for k in FIELDS):
                issues['malformed_row'] += 1; continue
            d, s, i = (row[k] for k in FIELDS[:3])
            if not all((d,s,i)): issues['empty_identifier'] += 1
            old = db.execute('select scene from records where dataset=? and image=?',(d,i)).fetchone()
            if old:
                issues['duplicate_composite_key'] += 1
                if old[0] != s: issues['inconsistent_scene'] += 1
            else:
                db.execute('insert into records values(?,?,?)',(d,i,s))
                counts[d,s] += 1
            poses[pose_state(row)] += 1
            if role in ('train','test'):
                candidates = [f'{role}/{d}/{s}/{i}', f'{role}/{d}/{i}']
                if any('..' in Path(p).parts or Path(i).is_absolute() for p in candidates):
                    issues['unsafe_image_reference'] += 1
                else:
                    found = db.execute('select count(*) from files where path in (?,?)',candidates).fetchone()[0]
                    if not found: issues['unresolved_image_reference'] += 1
    unique = db.execute('select count(*) from records').fetchone()[0]
    missing_test = extra_test = None
    if role == 'test':
        missing_test = db.execute("select count(*) from images i where role='test' and not exists(select 1 from records r where r.dataset=i.dataset and r.image=i.image)").fetchone()[0]
        extra_test = db.execute("select count(*) from records r where not exists(select 1 from images i where i.role='test' and r.dataset=i.dataset and r.image=i.image)").fetchone()[0]
    return {'status':'parsed','columns':h,'rows':rows,'unique_dataset_image_keys':unique,'complete':complete,
            'pose_states':dict(poses),'issues':dict(issues),'sample_keys_missing_visible_test_images':missing_test,
            'sample_keys_without_visible_test_image':extra_test,
            'identifier_format':'opaque image string; uniqueness key is (dataset,image)'}, counts

def audit(input_root='/kaggle/input', output='/kaggle/working/imc_audit', input_path=None, train_csv=None, sample_csv=None):
    out = Path(output).resolve(); out.mkdir(parents=True, exist_ok=True)
    if out.is_relative_to(Path(input_root).resolve()): raise ValueError('Outputs must be outside read-only inputs')
    probe = out/'.write_probe'; probe.write_text('ok'); probe.unlink()
    artifacts = []
    def save(name, value):
        dump(out/name,value); artifacts.append(name)
    save('execution_environment.json',{'utc_timestamp':datetime.now(timezone.utc).isoformat(),'python':platform.python_version(),
         'numpy':np.__version__,'platform':platform.system(),'gpu_required':False,'network_used':False,'output_write_probe':'passed'})
    status = {'state':'STARTED','real_kaggle_execution':str(input_root)=='/kaggle/input' and Path('/kaggle/working').is_dir(),
              'statistics_complete':False,'split_frozen':False,'limitations':[]}
    try:
        root,candidates,method = discover(input_root,input_path)
        status.update(input_root=str(root),discovery=method,candidates=candidates)
        inventory, metadata, archives = [], [], []
        total_bytes = entries = 0
        scan_complete = True
        with tempfile.TemporaryDirectory(dir=out) as temp, closing(sqlite3.connect(str(Path(temp)/'index.sqlite'))) as db:
            db.execute('create table files(path text primary key)')
            db.execute('create table images(role text,dataset text,image text)')
            db.execute('create index image_lookup on images(role,dataset,image)')
            for directory, dirs, names in os.walk(root,followlinks=False):
                base = Path(directory)
                dirs[:] = [d for d in dirs if not (base/d).is_symlink()]
                if len(inventory)<MAX_INVENTORY_RECORDS:
                    inventory.append({'path':base.relative_to(root).as_posix(),'type':'directory','bytes':None})
                for name in names:
                    p=base/name
                    if p.is_symlink(): continue
                    if entries>=MAX_FILES: scan_complete=False;break
                    rel=p.relative_to(root).as_posix();size=p.stat().st_size;ext=p.suffix.lower()
                    entries+=1;total_bytes+=size
                    db.execute('insert into files values(?)',(rel,))
                    if len(inventory)<MAX_INVENTORY_RECORDS: inventory.append({'path':rel,'type':ext or 'file','bytes':size})
                    parts=Path(rel).parts
                    if ext in ('.jpg','.jpeg','.png','.webp','.tif','.tiff') and len(parts)>=3 and parts[0] in ('train','test'):
                        db.execute('insert into images values(?,?,?)',(parts[0],parts[1],'/'.join(parts[2:]) if parts[0]=='test' else parts[-1]))
                    if ext in ('.csv','.json','.txt') and len(metadata)<1000:
                        item={'path':rel,'bytes':size,'sha256':None,'columns':None}
                        if size<=MAX_METADATA_BYTES and len(metadata)<200:
                            if ext=='.csv': item['columns']=header(p)
                            with p.open('rb') as f:item['sha256']=hashlib.file_digest(f,'sha256').hexdigest()
                        metadata.append(item)
                    if ext in ('.zip','.tar','.gz','.7z') and len(archives)<1000:
                        item={'path':rel,'bytes':size,'inspection':'not inspected; no extraction'}
                        if ext=='.zip' and size<=64*1024**2:
                            try:
                                with zipfile.ZipFile(p) as z:
                                    members=z.infolist();item.update(inspection='central directory only',members=len(members),expanded_bytes=sum(m.file_size for m in members),member_name_sample=[m.filename for m in members[:100]])
                            except zipfile.BadZipFile:item['inspection']='invalid ZIP'
                        archives.append(item)
                if not scan_complete:break
            db.commit()
            save('dataset_inventory.json',{'root':str(root),'scanned_files':entries,'reported_bytes':total_bytes,'scan_complete':scan_complete,
                 'inventory_records_complete':len(inventory)<MAX_INVENTORY_RECORDS,'records':inventory,'metadata':metadata,'archives':archives,
                 'metadata_catalog_may_be_capped':len(metadata)==1000,'archive_catalog_may_be_capped':len(archives)==1000,
                 'note':'Inventory includes path metadata only. Directory bytes null; total counts each file once. Large archives not opened.'})
            def choose(override, word):
                if override:
                    p=(root/override).resolve()
                    if not p.is_relative_to(root) or not p.is_file():raise ValueError('CSV override must name a file inside selected root')
                    return p
                options=[root/m['path'] for m in metadata if m['columns']==FIELDS and word in Path(m['path']).name.lower()]
                return options[0] if len(options)==1 else None
            labels=choose(train_csv,'train');sample=choose(sample_csv,'sample')
            train_result,counts=analyze_csv(labels,db,'train') if labels else ({'status':'unavailable_or_ambiguous','complete':False},Counter())
            sample_result,_=analyze_csv(sample,db,'test') if sample else ({'status':'unavailable_or_ambiguous','complete':False},Counter())
            train_result['source']=str(labels.relative_to(root)) if labels else None
            sample_result['source']=str(sample.relative_to(root)) if sample else None
            sample_result['pose_representation']='nine row-major rotation values and three translation values, separated by semicolons'
            sample_result['coverage_limit']='Visible test files only; archive-only or unrecognized layouts make unresolved references inconclusive. Sample poses are not ground truth.'
            save('pose_validation.json',train_result);save('submission_contract.json',sample_result)
            groups=Counter();scenes=Counter();outs=Counter()
            for (d,s),n in counts.items():
                groups[d]+=n
                if s=='outliers':outs[d]+=n
                else:scenes[d]+=1
            def table(name,columns,rows):
                with (out/name).open('w',newline='',encoding='utf-8') as f:
                    w=csv.writer(f);w.writerow(columns);w.writerows(rows)
                artifacts.append(name)
            table('dataset_groups.csv',['dataset','unique_images','scenes','outliers'],[(d,n,scenes[d],outs[d]) for d,n in sorted(groups.items())])
            table('scene_statistics.csv',['dataset','scene','unique_images'],[(d,s,n) for (d,s),n in sorted(counts.items())])
            table('outlier_statistics.csv',['dataset','outliers','unique_images','percent'],[(d,outs[d],n,100*outs[d]/n) for d,n in sorted(groups.items())])
            distribution=Counter(n for (d,s),n in counts.items() if s!='outliers')
            visible={role:dict(db.execute('select dataset,count(*) from images where role=? group by dataset',(role,))) for role in ('train','test')}
            save('dataset_summary.json',{'training_datasets':len(groups) if labels else None,'training_scenes':sum(scenes.values()) if labels else None,
                'unique_training_images':sum(groups.values()) if labels else None,'training_outliers':sum(outs.values()) if labels else None,
                'scene_size_distribution':dict(distribution),'visible_image_files_by_dataset':visible,'independent_groups':'UNKNOWN: dataset names are not proof of independence',
                'metadata_statistics_complete':train_result.get('complete',False),'file_scan_complete':scan_complete})
            thresholds=[]
            for m in metadata:
                if m['columns'] and {'dataset','scene','thresholds'}.issubset(m['columns']) and m['bytes']<=MAX_METADATA_BYTES:
                    values=[]
                    with (root/m['path']).open(newline='',encoding='utf-8-sig') as f:
                        for j,r in enumerate(csv.DictReader(f)):
                            if j>=10000:break
                            try:
                                ts=[float(x) for x in r['thresholds'].split(';')]
                                valid=bool(ts) and all(np.isfinite(x) and x>0 for x in ts)
                            except (ValueError,TypeError):ts=[];valid=False
                            values.append({'dataset':r.get('dataset'),'scene':r.get('scene'),'thresholds':ts if valid else None,'valid':valid})
                    thresholds.append({'source':m['path'],'sha256':m['sha256'],'rows':values,'complete':len(values)<10000,'units':'not inferred'})
            save('threshold_provenance.json',{'sources':thresholds,'synthetic_substitution':False,'official_parity':'not established'})
            bars=''.join(f'<text x="0" y="{j*22+18}">{n} images: {freq} scenes</text>' for j,(n,freq) in enumerate(sorted(distribution.items())[:100]))
            (out/'scene_sizes.svg').write_text(f'<svg xmlns="http://www.w3.org/2000/svg" width="500" height="{max(40,min(100,len(distribution))*22+20)}">{bars}</svg>')
            artifacts.append('scene_sizes.svg')
            review=f'''# Validation review\nObserved labeled dataset names: {len(groups) if labels else 'unavailable'}. Independent source groups: UNKNOWN.\nThe 60/20/20 allocation is NOT approved or frozen. With {len(groups)} named groups, each 20% partition would contain approximately {max(1,len(groups)//5) if groups else 0} groups; inspect dataset_groups.csv, outlier_statistics.csv and scene_statistics.csv for imbalance.\nBefore assignment, audit source provenance, exact duplicates and near-duplicates across groups. No image-content duplicate scan was run. Names and sizes cannot establish independence.\nIf few independent groups remain, consider leave-one-independent-group-out development CV, with a separately protected confirmation set if feasible. Its compute cost needs a later baseline pilot; feasibility is not established by this audit. Scene-disjoint diagnostics can complement dataset-level generalization but cannot replace it. Small scenes remain reported; do not exclude them post hoc.\nFreeze baseline, data, evaluator, thresholds and conditions once approved; every improvement uses that same version. Never tune on confirmation. Official prose mAA-first assignment remains distinct from participant overlap-first code; no parity claim.\n'''
            (out/'validation_recommendation.md').write_text(review);artifacts.append('validation_recommendation.md')
            status.update(state='AUDITED',statistics_complete=scan_complete and train_result.get('complete',False),
                limitations=['No source-independence or near-duplicate audit','No hidden test labels accessed','Image-reference resolution supports train/dataset/scene/image and test/dataset/image; unresolved is not proof of missing content','Archive extraction not performed'])
    except (OSError,ValueError,csv.Error,UnicodeError,sqlite3.Error) as exc:
        status.update(state='BLOCKED',error_type=type(exc).__name__,message=str(exc))
    save('audit_status.json',status)
    # Explicit allowlist: never archive input files, SQLite indices, or old outputs.
    with zipfile.ZipFile(out/'imc_dataset_audit.zip','w',zipfile.ZIP_DEFLATED) as archive:
        used=0
        included=[]; omitted=[]
        for name in artifacts:
            size=(out/name).stat().st_size
            if size>10*1024**2 or used+size>25*1024**2:
                omitted.append(name);continue
            archive.write(out/name,name);used+=size
            included.append({'file':name,'bytes':size,'sha256':hashlib.sha256((out/name).read_bytes()).hexdigest()})
        archive.writestr('archive_manifest.json',json.dumps({'included':included,'omitted_due_to_size_cap':omitted},indent=2))
    return status


"""Offline CPU Phase 2C audit; no predictions or split generation."""
import csv
import hashlib
import json
import math
import os
import platform
import re
import sqlite3
import time
import zipfile
from collections import Counter, defaultdict
from contextlib import closing
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
from PIL import Image, ImageOps

TRAIN_FIELDS=['dataset','scene','image','rotation_matrix','translation_vector']
SUBMISSION_FIELDS=['image_id']+TRAIN_FIELDS
EXPECTED_THRESHOLD_HASH='5b92db83d510ab19e66bd1ef4c23865245d75b4a981986e6d8eb1aee06fc9fe2'

def sha(path):
    with Path(path).open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()

def write_json(path,value):
    Path(path).write_text(json.dumps(value,indent=2,allow_nan=False),encoding='utf-8')

def csv_rows(path, fields):
    with Path(path).open(newline='',encoding='utf-8-sig') as f:
        r=csv.DictReader(f)
        if r.fieldnames!=fields:raise ValueError(f'Expected schema {fields}, observed {r.fieldnames}')
        for index,row in enumerate(r):
            if None in row or any(v is None for v in row.values()):raise ValueError(f'Malformed CSV row {index}')
            yield index,row

def write_table(path,fields,rows):
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        w=csv.writer(f);w.writerow(fields);w.writerows(rows)

def validate_submission(sample, output, visible=None):
    """Preserve strings and row order in a private-to-run mapping, excluded from ZIP."""
    out=Path(output);out.mkdir(parents=True,exist_ok=True)
    issues=Counter();counts=Counter();scenes=Counter();forms=Counter();types=Counter()
    ids=set();keys=set();filenames=defaultdict(set);missing_visible=0;rows=0
    with (out/'sample_row_identity.csv').open('w',newline='',encoding='utf-8') as f:
        writer=csv.writer(f);writer.writerow(['row_index']+SUBMISSION_FIELDS)
        for index,row in csv_rows(sample,SUBMISSION_FIELDS):
            rows+=1;writer.writerow([index]+[row[k] for k in SUBMISSION_FIELDS])
            ident=row['image_id'];key=(row['dataset'],row['image'])
            if not ident.strip():issues['empty_image_id']+=1
            if not all(v.strip() for v in key):issues['empty_dataset_or_image']+=1
            if ident in ids:issues['duplicate_image_id']+=1
            if key in keys:issues['duplicate_dataset_image']+=1
            ids.add(ident);keys.add(key);counts[key[0]]+=1;filenames[Path(key[1]).name].add(key[0])
            scenes[row['scene']]+=1
            for k,v in row.items():
                kind='empty' if v=='' else 'integer_like_string' if re.fullmatch(r'[+-]?\d+',v) else 'string'
                types[k,kind]+=1
            try:
                r=[float(x) for x in row['rotation_matrix'].split(';')];t=[float(x) for x in row['translation_vector'].split(';')]
                if len(r)!=9 or len(t)!=3:raise ValueError()
                forms['all_nan' if all(math.isnan(x) for x in r+t) else 'numeric_vectors']+=1
            except ValueError:forms['malformed_vector_encoding']+=1
            if visible is not None and key not in visible:missing_visible+=1
    result={'source_sha256':sha(sample),'columns':SUBMISSION_FIELDS,'rows':rows,'unique_image_id':len(ids),'unique_dataset_image':len(keys),
            'unique_image_strings':len({i for _,i in keys}),'issues':dict(issues),'per_dataset':dict(counts),
            'column_types':'CSV strings preserved without coercion','observed_lexical_types':{f'{k}:{v}':n for (k,v),n in types.items()},
            'scene_value_counts':dict(scenes),'pose_encoding_counts':dict(forms),'physical_pose_validation_applied':False,
            'placeholder_semantics':'Patterns reported only; numeric values or scene strings do not establish prediction meaning',
            'repeated_basenames_across_datasets':sum(len(v)>1 for v in filenames.values()),
            'sample_rows_without_visible_image':missing_visible if visible is not None else None,
            'visible_test_keys_absent_from_sample':len(visible-keys) if visible is not None else None,
            'visible_coverage_note':'Unmatched rows are not proof of invalid submission or hidden-test size. Visible inputs only.',
            'row_identity_mapping':'sample_row_identity.csv (internal; excluded from portable ZIP)','complete':True,'scorer_acceptance':'UNVERIFIED'}
    write_json(out/'sample_submission_contract.json',result)
    write_table(out/'sample_dataset_coverage.csv',['dataset','sample_rows','visible_test_keys'],[(d,n,sum(x==d for x,_ in visible) if visible is not None else '') for d,n in sorted(counts.items())])
    return result

def threshold_tokens(raw):
    result=[]
    for token in raw.split(';'):
        value=None
        try:
            number=float(token)
            kind='nonfinite' if not math.isfinite(number) else 'zero' if number==0 else 'negative' if number<0 else 'positive'
            value=number if math.isfinite(number) else None
        except ValueError:kind='missing' if not token.strip() else 'unparsed_encoding'
        result.append({'raw':token,'type':kind,'finite_value':value})
    return result

def thresholds(path,out):
    digest=sha(path);rows=[]
    with Path(path).open(newline='',encoding='utf-8-sig') as f:
        reader=csv.DictReader(f);columns=reader.fieldnames
        if not {'dataset','scene','thresholds'}.issubset(columns or []):raise ValueError('Unrecognized threshold schema')
        for idx,row in enumerate(reader):
            if row.get('thresholds') is None:raise ValueError('Missing threshold field')
            tokens=threshold_tokens(row['thresholds']);reasons=sorted({t['type'] for t in tokens if t['type']!='positive'})
            rows.append([idx,row.get('dataset'),row.get('scene'),row['thresholds'],json.dumps(tokens),';'.join(reasons),json.dumps(row)])
    write_table(Path(out)/'thresholds_raw_and_parsed.csv',['row_index','dataset','scene','raw_thresholds','parsed_tokens_json','previous_positive_validator_rejection','original_row_json'],rows)
    info={'sha256':digest,'expected_sha256':EXPECTED_THRESHOLD_HASH,'hash_matches_previous':digest==EXPECTED_THRESHOLD_HASH,'columns':columns,'rows':len(rows),
          'interpretation':'UNRESOLVED: preserve raw tokens; participant parser uses float(token) on semicolon-separated strings. A representable float is not proof of official intended semantics.',
          'configuration_created':False,'complete':True}
    write_json(Path(out)/'threshold_source.json',info)
    target=[r for r in rows if r[1]=='imc2023_haiper' and r[2] in ('bike','chairs','fountain')]
    (Path(out)/'threshold_reconciliation.md').write_text('# Threshold inspection\n\n'+json.dumps(info,indent=2)+'\n\nHaiper raw diagnostics (no substituted values):\n'+json.dumps(target,indent=2)+'\n\nA failed positivity check does not prove invalid metadata. Review raw tokens and actual evaluator behavior before creating an executable threshold configuration. The source was not modified.\n')
    return info

def pose_category(row):
    try:
        r=np.array([float(x) for x in row['rotation_matrix'].split(';')]).reshape(3,3);t=np.array([float(x) for x in row['translation_vector'].split(';')]).reshape(3)
        x=np.r_[r.ravel(),t]
        if np.isnan(x).all():return 'fully_nan',False
        if np.isfinite(x).all():
            ok=np.allclose(r.T@r,np.eye(3),atol=1e-3,rtol=0) and abs(np.linalg.det(r)-1)<=1e-3
            return 'fully_finite',bool(ok)
        if np.isnan(x).any() and not np.isinf(x).any():return 'mixed_nan_finite',False
        return 'other_invalid',False
    except ValueError:return 'other_invalid',False

def pose_audit(path,out):
    counts=Counter();bad_rot=Counter();scenes=defaultdict(set);n=0
    for _,row in csv_rows(path,TRAIN_FIELDS):
        n+=1;category,rotation_ok=pose_category(row);flag=row['scene']=='outliers'
        counts[row['dataset'],flag,category]+=1;scenes[row['scene']].add(row['dataset'])
        if category=='fully_finite' and not rotation_ok:bad_rot[row['dataset']]+=1
    write_table(Path(out)/'pose_outlier_crosstab.csv',['dataset','is_outlier','pose_state','count'],[(d,o,s,n) for (d,o,s),n in sorted(counts.items())])
    result={'rows':n,'source_sha256':sha(path),'rotation_atol':1e-3,'rotation_rtol':0,'determinant_abs_tolerance':1e-3,
        'finite_invalid_rotations_by_dataset':dict(bad_rot),'nonoutlier_all_nan':sum(n for (d,o,s),n in counts.items() if not o and s=='fully_nan'),
        'outlier_not_all_nan':sum(n for (d,o,s),n in counts.items() if o and s!='fully_nan'),'complete':True}
    write_json(Path(out)/'pose_integrity.json',result);return scenes

def constrained_groups(datasets, exact_edges):
    parent={d:d for d in datasets}
    def find(d):
        while parent[d]!=d:d=parent[d]
        return d
    for a,b in exact_edges:parent[find(b)]=find(a)
    groups=defaultdict(list)
    for d in sorted(datasets):groups[find(d)].append(d)
    return sorted(groups.values())

def overlap(root,out,seconds=120,max_images=10000,near_distance=5):
    """Train-only byte hashes and 64-bit thumbnail dHash. Resume via versioned cache."""
    start=time.monotonic();deadline=start+seconds;root=Path(root);out=Path(out)
    files=[];scan_complete=True;warnings=[]
    for directory,dirs,names in os.walk(root/'train',followlinks=False):
        dirs[:]=[d for d in dirs if not (Path(directory)/d).is_symlink()]
        for name in names:
            p=Path(directory)/name
            if p.is_symlink() or p.suffix.lower() not in ('.jpg','.jpeg','.png','.webp','.tif','.tiff'):continue
            if time.monotonic()>deadline or len(files)>=max_images:scan_complete=False;break
            parts=p.relative_to(root/'train').parts
            if len(parts)>=2:files.append((p,parts[0],p.stat().st_size,p.stat().st_mtime_ns))
        if not scan_complete:break
    files.sort(key=lambda x:x[0].as_posix())
    catalog_hash=hashlib.sha256(json.dumps([(str(p.relative_to(root)),d,s,m) for p,d,s,m in files]).encode()).hexdigest()
    cache_path=out/'overlap_cache.json';cache={}
    if cache_path.exists():
        old=json.loads(cache_path.read_text())
        if old.get('catalog_hash')==catalog_hash and old.get('algorithm')=='dhash64-exif-gray-v1':cache=old['entries']
    size_groups=defaultdict(set)
    for p,d,size,m in files:size_groups[size].add(d)
    exact_needed=sum(len(size_groups[s])>1 for _,_,s,_ in files)
    errors=[]
    for p,d,size,m in files:
        if time.monotonic()>deadline:break
        key=p.relative_to(root).as_posix();entry=cache.setdefault(key,{})
        try:
            if len(size_groups[size])>1 and 'sha256' not in entry:
                h=hashlib.sha256()
                with p.open('rb') as f:
                    while True:
                        if time.monotonic()>deadline:raise TimeoutError()
                        chunk=f.read(1024*1024)
                        if not chunk:break
                        h.update(chunk)
                entry['sha256']=h.hexdigest()
            if 'dhash' not in entry:
                with Image.open(p) as im:
                    im.draft('RGB',(128,128));im.thumbnail((128,128))
                    small=ImageOps.exif_transpose(im).convert('L').resize((9,8))
                    arr=np.asarray(small);bits=(arr[:,1:]>arr[:,:-1]).ravel()
                    entry['dhash']=int(''.join('1' if b else '0' for b in bits),2)
        except TimeoutError:break
        except (OSError,ValueError,Image.DecompressionBombError) as exc:errors.append({'path':key,'error_type':type(exc).__name__})
    write_json(cache_path,{'catalog_hash':catalog_hash,'algorithm':'dhash64-exif-gray-v1','entries':cache})
    candidates=[];edges=set();comparisons=0;pair_complete=True
    dataset_counts=Counter(d for _,d,_,_ in files)
    total_pairs=(len(files)**2-sum(n*n for n in dataset_counts.values()))//2
    normalized=lambda p:re.sub(r'[^a-z0-9]','',p.stem.lower())
    for i,(p,a,_,_) in enumerate(files):
        for q,b,_,_ in files[i+1:]:
            if a==b:continue
            if time.monotonic()>deadline or len(candidates)>=20000:pair_complete=False;break
            comparisons+=1;x=cache.get(p.relative_to(root).as_posix(),{});y=cache.get(q.relative_to(root).as_posix(),{})
            kind=None;distance=None
            if x.get('sha256') and x.get('sha256')==y.get('sha256'):kind='exact_bytes';edges.add(tuple(sorted((a,b))))
            elif 'dhash' in x and 'dhash' in y:
                distance=(x['dhash']^y['dhash']).bit_count()
                if distance<=near_distance:kind='perceptual_candidate_UNREVIEWED'
            if kind is None and normalized(p)==normalized(q):kind='normalized_filename_only'
            if kind:candidates.append((a,b,p.relative_to(root).as_posix(),q.relative_to(root).as_posix(),kind,distance))
        if not pair_complete:break
    write_table(out/'dataset_overlap_candidates.csv',['dataset_a','dataset_b','path_a','path_b','evidence','dhash_distance'],candidates)
    info={'inventory_images':len(files),'inventory_complete':scan_complete,'inventory_catalog_sha256':catalog_hash,'exact_hash_candidates':exact_needed,
          'exact_hashes_completed':sum('sha256' in x for x in cache.values()),'thumbnail_hashes_completed':sum('dhash' in x for x in cache.values()),
          'cross_dataset_pairs_compared':comparisons,'total_cross_dataset_pairs':total_pairs,'pair_scan_complete':pair_complete,
          'complete':scan_complete and pair_complete and sum('dhash' in x for x in cache.values())==len(files) and sum('sha256' in x for x in cache.values())==exact_needed,
          'decode_errors':errors,'elapsed_seconds':time.monotonic()-start,'budget_seconds':seconds,'budget_note':'Cooperative time limit; a single image decode/stat may overrun. JPEG draft/thumbnail; one image at a time. No test imagery read.',
          'near_distance':near_distance,'near_matches_manually_reviewed':False,'resume':'Rerun with same mounts/output; cache reused only for identical size/mtime/path catalog. Increase budget for pair stage; it restarts deterministically.',
          'exact_group_constraints':constrained_groups({d for _,d,_,_ in files},edges)}
    write_json(out/'overlap_status.json',info);return info

def run(input_root='/kaggle/input',output='/kaggle/working/imc_contract_audit',input_path=None,seconds=120,max_images=10000):
    out=Path(output).resolve();out.mkdir(parents=True,exist_ok=True)
    if out.is_relative_to(Path(input_root).resolve()):raise ValueError('Output cannot be under input')
    status={'state':'STARTED','split_frozen':False,'warnings':[]}
    names=['environment.json','audit_status.json']
    write_json(out/'environment.json',{'utc':datetime.now(timezone.utc).isoformat(),'python':platform.python_version(),'numpy':np.__version__,'network':False,'gpu':False})
    try:
        root,_,_=discover(input_root,input_path);status['root']=str(root)
        # Only visible test paths are enumerated; their image bytes are never read.
        visible=set()
        for p in (root/'test').rglob('*'):
            if p.is_file() and p.suffix.lower() in ('.jpg','.jpeg','.png','.webp','.tif','.tiff'):
                parts=p.relative_to(root/'test').parts
                if len(parts)>=2:visible.add((parts[0],'/'.join(parts[1:])))
        validate_submission(root/'sample_submission.csv',out,visible)
        names+=['sample_submission_contract.json','sample_dataset_coverage.csv']
        thresholds(root/'train_thresholds.csv',out);names+=['thresholds_raw_and_parsed.csv','threshold_source.json','threshold_reconciliation.md']
        scenes=pose_audit(root/'train_labels.csv',out);names+=['pose_outlier_crosstab.csv','pose_integrity.json']
        info=overlap(root,out,seconds,max_images);names+=['dataset_overlap_candidates.csv','overlap_status.json']
        shared={s:sorted(ds) for s,ds in scenes.items() if s!='outliers' and len(ds)>1}
        licenses=[]
        for p in (root/'train').rglob('*LICENSE*'):
            if p.is_file() and len(licenses)<100:
                with p.open(encoding='utf-8',errors='replace') as f:prefix=f.read(65536)
                licenses.append({'path':p.relative_to(root).as_posix(),'sha256':sha(p),'bytes':p.stat().st_size,'urls_in_first_64KiB':sorted(set(re.findall(r'https?://[^\s<>"\)]+',prefix)))[:20]})
        (out/'source_provenance_review.md').write_text('# Provenance evidence\nShared scene names (not proof of duplication):\n'+json.dumps(shared,indent=2)+'\nLicense/source metadata (URL sample, not a complete attribution audit):\n'+json.dumps(licenses,indent=2)+'\nExact byte matches require grouping. Perceptual candidates require human review of training images in Kaggle; no thumbnails exported. No matches does not establish independent sources.\n')
        (out/'validation_grouping_proposal.md').write_text('# DRAFT grouping; no split frozen\nMust-link groups from observed exact duplicates:\n'+json.dumps(info['exact_group_constraints'],indent=2)+'\nUncertain shared-scene groups:\n'+json.dumps(shared,indent=2)+'\nConservatively keep the two dioscuri datasets together pending provenance review even without byte duplicates. Historical IMC and PhotoTourism naming suggests source families but does not establish independence or mandatory merging. Inspect near candidates before approval. Outlier-heavy domains (heritage, dioscuri/baalshamin, lizard/pond, ETs) must inform representative development/confirmation coverage without model scores. Prefer grouped development folds; assess leave-one-group-out cost later. Scene-disjoint diagnostics are supplementary. Protect confirmation from tuning. Freeze identical baseline, data, evaluator, thresholds and conditions for all improvements only after supervisor approval.\n')
        names+=['source_provenance_review.md','validation_grouping_proposal.md']
        status.update(state='AUDITED',overlap_complete=info['complete'],threshold_interpretation='UNRESOLVED',submission_acceptance='UNVERIFIED')
        if not info['complete']:status['warnings'].append('PARTIAL_OVERLAP_AUDIT')
        status['warnings']+=['NEAR_MATCHES_UNREVIEWED','SOURCE_INDEPENDENCE_NOT_ESTABLISHED','THRESHOLD_SEMANTICS_REQUIRE_REVIEW']
    except (OSError,ValueError,csv.Error) as exc:status.update(state='BLOCKED',error_type=type(exc).__name__,message=str(exc))
    write_json(out/'audit_status.json',status)
    with zipfile.ZipFile(out/'imc_contract_audit.zip','w',zipfile.ZIP_DEFLATED) as z:
        manifest=[];omitted=[];size=0
        for name in names:
            p=out/name
            if p.stat().st_size>10*1024**2 or size+p.stat().st_size>25*1024**2:omitted.append(name);continue
            size+=p.stat().st_size;z.write(p,name);manifest.append({'file':name,'sha256':sha(p),'bytes':p.stat().st_size})
        z.writestr('archive_manifest.json',json.dumps({'included':manifest,'omitted':omitted},indent=2))
    return status


In [ ]:
INPUT_PATH = None  # Optional mounted root beneath /kaggle/input
MAX_OVERLAP_SECONDS = 120
MAX_TRAIN_IMAGES = 10000
status = run(input_path=INPUT_PATH, seconds=MAX_OVERLAP_SECONDS, max_images=MAX_TRAIN_IMAGES)
print(json.dumps(status, indent=2))
print("Download /kaggle/working/imc_contract_audit/imc_contract_audit.zip")


Review audit_status.json and overlap_status.json. Partial scans are not negative evidence. Rerun with a larger time budget in the same session to reuse the path/size/mtime-keyed cache; pair screening restarts. The identity mapping and cache stay in working storage and are excluded from ZIP. Review perceptual candidates manually against training images; no source images are exported. Threshold meaning and official acceptance remain unverified until supervisor review.
